In [1]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility (VERY IMPORTANT for academic projects)
np.random.seed(42)

# Load your final cluster dataset
clusters = pd.read_csv("final_cluster_features.csv")

# 333 users × 18 clusters = 5994 rows ≈ 6000 ✅ exactly what you requested
NUM_USERS = 333

In [2]:
# Generate random realistic users
users = []

for i in range(NUM_USERS):
    user = {
        'Likes_Beach': np.random.randint(0, 2),
        'Likes_Mountain': np.random.randint(0, 2),
        'Likes_Culture': np.random.randint(0, 2),
        'Likes_Adventure': np.random.randint(0, 2),
        'Budget': np.random.randint(1, 4),
        'Total_Days': np.random.randint(1, 8)
    }
    users.append(user)

users = pd.DataFrame(users)

In [3]:
# ✅ EXACT SCORING FORMULA AS DESIGNED
def calculate_score(user, cluster):

    # 1. Preference Score (0-70) MAIN DRIVER
    preference_score = 0
    preference_score += user['Likes_Beach'] * cluster['Beach'] * 20
    preference_score += user['Likes_Mountain'] * (cluster['Nature'] + cluster['Hiking']) * 20
    preference_score += user['Likes_Culture'] * (cluster['History'] + cluster['Religious'] + cluster['Culture']) * 20
    preference_score += user['Likes_Adventure'] * (cluster['Adventure'] + cluster['Safari']) * 10

    # 2. Distance Score (0-20) SMART LOGIC
    ideal_distance = user['Total_Days'] / 7
    distance_score = (1 - abs(cluster['Avg_Distance'] - ideal_distance)) * 20

    # 3. Budget Penalty (0 to -15)
    if user['Budget'] == 1:
        budget_penalty = cluster['Avg_Distance'] * 15
    else:
        budget_penalty = 0

    # 4. Small random noise (-5 to +5)
    noise = np.random.uniform(-5, 5)

    # Calculate final score
    score = preference_score + distance_score - budget_penalty + noise

    # Clip to 0-100 range
    return max(0, min(100, score))

In [4]:
# Generate full training dataset
training_data = []

for _, user in users.iterrows():
    for _, cluster in clusters.iterrows():

        score = calculate_score(user, cluster)

        row = {
            # User features
            'Likes_Beach': user['Likes_Beach'],
            'Likes_Mountain': user['Likes_Mountain'],
            'Likes_Culture': user['Likes_Culture'],
            'Likes_Adventure': user['Likes_Adventure'],
            'Budget': user['Budget'],
            'Total_Days': user['Total_Days'],

            # Cluster features
            'Cluster_Name': cluster['Cluster_Name'],
            'Adventure': cluster['Adventure'],
            'Architecture': cluster['Architecture'],
            'Beach': cluster['Beach'],
            'Birding': cluster['Birding'],
            'Culture': cluster['Culture'],
            'Hiking': cluster['Hiking'],
            'History': cluster['History'],
            'Museum': cluster['Museum'],
            'Nature': cluster['Nature'],
            'Park': cluster['Park'],
            'Relax': cluster['Relax'],
            'Religious': cluster['Religious'],
            'Safari': cluster['Safari'],
            'Shopping': cluster['Shopping'],
            'Viewpoint': cluster['Viewpoint'],
            'Num_Places': cluster['Num_Places'],
            'Avg_Distance': cluster['Avg_Distance'],

            # Target output
            'Score': score
        }

        training_data.append(row)

training_data = pd.DataFrame(training_data)

In [5]:
print(f"Generated dataset size: {len(training_data)} rows")

# Save 2 versions:

# 1. Full version with Cluster_Name (for debugging and checking)
training_data.to_csv("training_data_full.csv", index=False)

# 2. Clean version for ML training (remove Cluster_Name)
training_ml = training_data.drop(columns=['Cluster_Name'])
training_ml.to_csv("training_data_ml.csv", index=False)

Generated dataset size: 5994 rows


In [6]:
# Test: Show average score for beach lovers
beach_user = training_data[training_data['Likes_Beach'] == 1]
print("Average score for beach user:")
print(beach_user.groupby('Cluster_Name')['Score'].mean().sort_values(ascending=False).head(5))

Average score for beach user:
Cluster_Name
Matara Area        24.373408
Trincomalee        21.330962
Galle Area         21.146000
Batticaloa Area    19.781309
Dambulla Area      19.720005
Name: Score, dtype: float64
